# Mask-Aware Liveness / Anti-Spoofing Access Gate — Demo Analysis

This notebook loads the artifacts produced by `train.py` and `evaluate.py` and
presents them for the CSE471 report: training curves, confusion matrix,
APCER/BPCER/HTER metrics, and a few sample predictions.

Run `train.py` and `evaluate.py` from `src/` before running this notebook.

In [ ]:
import json
import os
import sys

import pandas as pd
from IPython.display import Image, display

sys.path.append(os.path.abspath("../src"))
import config

## 1. Training curves

In [ ]:
curves_path = os.path.join(config.PLOTS_DIR, "training_curves.png")
if os.path.exists(curves_path):
    display(Image(filename=curves_path))
else:
    print("Run train.py first to generate training_curves.png")

In [ ]:
history_path = os.path.join(config.PLOTS_DIR, "training_history.json")
if os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)
    pd.DataFrame(history)
else:
    print("Run train.py first to generate training_history.json")

## 2. Test-set evaluation: CelebA-Spoof held-out test split

In [ ]:
metrics_path = os.path.join(config.PREDICTIONS_DIR, "metrics_test.json")
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)
    pd.DataFrame([metrics]).T.rename(columns={0: "value"})
else:
    print("Run: python evaluate.py --data_root ../data/celeba_spoof/test --tag test")

In [ ]:
cm_path = os.path.join(config.PLOTS_DIR, "confusion_matrix_test.png")
if os.path.exists(cm_path):
    display(Image(filename=cm_path))

## 3. Generalization check: CASIA-FASD

CASIA-FASD was never used for training -- it only tests how well the model
generalizes to a completely different camera/attack distribution.

In [ ]:
casia_metrics_path = os.path.join(config.PREDICTIONS_DIR, "metrics_casia_fasd.json")
if os.path.exists(casia_metrics_path):
    with open(casia_metrics_path) as f:
        casia_metrics = json.load(f)
    pd.DataFrame([casia_metrics]).T.rename(columns={0: "value"})
else:
    print("Run: python evaluate.py --data_root ../data/casia_fasd/test --tag casia_fasd")

## 4. Sample predictions (highest-confidence correct and incorrect examples)

In [ ]:
preds_path = os.path.join(config.PREDICTIONS_DIR, "predictions_test.csv")
if os.path.exists(preds_path):
    df = pd.read_csv(preds_path)
    df["correct"] = df["true_label"] == df["pred_label"]
    print("Total:", len(df), " Correct:", df["correct"].sum(), " Wrong:", (~df["correct"]).sum())
    display(df[~df["correct"]].sort_values("prob_spoof", ascending=False).head(10))
else:
    print("Run evaluate.py first to generate predictions_test.csv")

## 5. Stage 1 sanity check: mask-aware face detection

Run `python test_masked_samples.py` in `src/` first, then view the annotated
images it writes to `outputs/predictions/masked_detection_check/`.

In [ ]:
masked_dir = os.path.join(config.PREDICTIONS_DIR, "masked_detection_check")
if os.path.isdir(masked_dir):
    for fname in sorted(os.listdir(masked_dir))[:5]:
        display(Image(filename=os.path.join(masked_dir, fname)))
else:
    print("Run test_masked_samples.py first.")